# ETL Pipeline – Slutuppgift Data Science

Denna notebook implementerar en ETL-pipeline som körs på två dataset:
- main dataset
- validation dataset


In [1]:
!pip install pandas numpy python-dotenv matplotlib seaborn langchain langchain_groq

  Using cached numpy-2.4.1-cp314-cp314-macosx_14_0_x86_64.whl.metadata (6.6 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-macosx_10_13_x86_64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached langchain-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_groq-1.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_10_13_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp314-cp314-macosx_10_15_x86_64.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp314-cp314-macosx_10_13_x86_64.whl.metadata (6.3 kB)
  Using cached langchain_core-1.2.7-py3-none-any.whl.metadata (3.7 kB)
  Using cached langgraph-1.0.6-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-no

# IMPORTERA BIBLOTEK OCH LLM

Här kommer alla importer jag behöver för att kunna slutföra uppgiften.
Även här så kommer jag att ha min LLM.

In [2]:
import os
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "Saknar GROQ_API_KEY i .env"


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

if llm:
    print("LLM is initialized")

/Users/rikardsoderstrom/DataScience-uppgift/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


LLM is initialized


# Ladda in och inspektera dataset


In [10]:
df_raw = pd.read_csv("nordtech_data.csv")

df_raw.head()

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
0,ORD-2024-00001,ORD-2024-00001-1,2024-05-19,2024-05-22,SKU-WC001,Webbkamera HD,Tillbehör,1,SEK 799,Uppsala,Privat,Kort,KND-53648,Levererad,NaN,NaN,NaN
1,ORD-2024-00002,ORD-2024-00002-1,2024-12-02,5 december 2024,SKU-HB001,USB-C Hub 7-port,Tillbehör,1,549.00,Göteborg,Privat,Swish,KND-84095,Levererad,NaN,NaN,NaN
2,ORD-2024-00003,ORD-2024-00003-1,2024-12-31,2025-01-03,SKU-SD001,Extern SSD 1TB,Lagring,1,1199.00,NaN,Företag,Faktura,KND-91748,Levererad,Stämmer inte överens med produktbeskrivningen.,2025-01-12,2.0
3,ORD-2024-00003,ORD-2024-00003-2,2024-12-31,2025-01-03,SKU-SD002,Extern SSD 500GB,Lagring,10,699 kr,Stockholm,Företag,FAKTURA,KND-91748,Mottagen,"Leveransen tog lite längre än utlovat, men pro...",2025-01-14,3.0
4,ORD-2024-00003,ORD-2024-00003-3,2024-12-31,2025-01-03,SKU-MS001,Trådlös Mus X1,Tillbehör,1,399.00,Stockholm,Företag,Faktura,KND-91748,NaN,NaN,NaN,NaN


In [11]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2767 entries, 0 to 2766
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         2767 non-null   str    
 1   orderrad_id      2767 non-null   str    
 2   orderdatum       2767 non-null   str    
 3   leveransdatum    2767 non-null   str    
 4   produkt_sku      2767 non-null   str    
 5   produktnamn      2767 non-null   str    
 6   kategori         2767 non-null   str    
 7   antal            2767 non-null   str    
 8   pris_per_enhet   2767 non-null   str    
 9   region           2612 non-null   str    
 10  kundtyp          2767 non-null   str    
 11  betalmetod       2651 non-null   str    
 12  kund_id          2767 non-null   str    
 13  leveransstatus   2673 non-null   str    
 14  recension_text   1355 non-null   str    
 15  recensionsdatum  1355 non-null   str    
 16  betyg            1355 non-null   float64
dtypes: float64(1), str(16)
me

In [12]:
df_raw.tail()

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
2762,ORD-2024-01654,ORD-2024-01654-3,2024-10-04,2024-10-07,SKU-LP003,Laptop Gaming X,Datorer,2,18999.00,stockholm,Företag,NaN,KND-99742,Levererad,"Förväntade mig mer för priset, men den duger.",2024-10-08,3.0
2763,ORD-2024-01655,ORD-2024-01655-1,2024-01-15,2024-01-17,SKU-MN003,"Bildskärm 32"" Curved",Bildskärmar,1,5999.00,Göteborg,Privat,Faktura,KND-83827,Returnerad,Stämmer inte överens med produktbeskrivningen.,2024-01-27,2.0
2764,ORD-2024-01656,ORD-2024-01656-1,2024-07-29,2024-07-31,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,göteborg,Privat,faktura,KND-60471,Levererad,NaN,NaN,NaN
2765,ORD-2024-01656,ORD-2024-01656-2,2024-07-29,2024-07-31,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Göteborg,Konsument,Faktura,KND-60471,Levererad,NaN,NaN,NaN
2766,ORD-2024-01657,ORD-2024-01657-1,2024-07-03,2024-07-05,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,STOCKHOLM,Privat,NaN,KND-26325,levererad,Överträffade mina förväntningar. 5 av 5!,2024-07-08,4.0


In [13]:
df_raw.sample(10)

,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
2010,ORD-2024-01205,ORD-2024-01205-1,2024-05-06,2024-05-08,SKU-HS001,Headset Pro ANC,Ljud,2,SEK 1899,Stockholm,Privat,Kort,KND-41689,Levererad,NaN,NaN,NaN
855,ORD-2024-00531,ORD-2024-00531-2,2024-07-16,2024-07-19,SKU-MN003,"Bildskärm 32"" Curved",Bildskärmar,1,5999 kr,UPPSALA,Privat,Swish,KND-61712,Levererad,"Leverans på två dagar, imponerad!",28 juli 2024,5.0
2746,ORD-2024-01646,ORD-2024-01646-2,"February 01, 2024",2024-02-04,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Stockholm,Privat,Faktura,KND-92109,Levererad,Medelmåttig upplevelse.,8 februari 2024,3.0
604,ORD-2024-00375,ORD-2024-00375-2,2024-01-05,2024-01-08,SKU-LP001,Laptop Pro 15,Datorer,1,14999.00,Stockholm,Privat,Swish,KND-44657,Levererad,NaN,NaN,NaN
2372,ORD-2024-01418,ORD-2024-01418-2,2024-07-02,2024-07-05,SKU-HS002,Headset Budget,Ljud,1,499.00,göteborg,PRIVAT,Kort,KND-27754,Levererad,NaN,NaN,NaN
1926,ORD-2024-01159,ORD-2024-01159-1,14 december 2024,2024-12-17,SKU-SD002,Extern SSD 500GB,Lagring,1,699.00,ÖREBRO,Privat,Kort,KND-69711,Levererad,Toppen! Exakt vad jag behövde.,2024-12-21,4.0
789,ORD-2024-00489,ORD-2024-00489-2,2024-02-19,2024-02-22,SKU-HS002,Headset Budget,Ljud,1,499.00,Stockholm,Företag,Faktura,KND-83717,Levererad,NaN,NaN,NaN
2585,ORD-2024-01547,ORD-2024-01547-1,2024-12-08,2024-12-11,SKU-MN001,"Bildskärm 27"" UHD",Bildskärmar,1,4999.00,Örebro,PRIVAT,Faktura,KND-81067,Levererad,NaN,NaN,NaN
402,ORD-2024-00250,ORD-2024-00250-1,2024-11-20,2024-11-17,SKU-KB001,Mekaniskt Tangentbord K7,Tillbehör,1,1299.00,Malmö,privat,Faktura,KND-56094,Levererad,NaN,NaN,NaN
1938,ORD-2024-01166,ORD-2024-01166-3,2024/02/03,2024-02-06,SKU-SP001,Bluetooth-högtalare,Ljud,1,899.00,Göteborg,Privat,Kort,KND-38153,Levererad,NaN,NaN,NaN


In [38]:
df_raw["region"].value_counts()

region
Stockholm     834
Göteborg      434
Malmö         236
Uppsala       192
Norrland      125
Örebro        117
Linköping     117
Västerås       75
STOCKHOLM      50
Sthml          44
stockholm      42
STHLM          39
uppsala        25
Sthlm          25
göteborg       24
GÖTEBORG       22
UPPSALA        21
Gothenburg     19
MALMÖ          16
Gbg            14
malmo          12
LINKÖPING      11
GBGB           11
Orebro         10
Vasteras       10
örebro          9
ÖREBRO          9
norrland        9
linköping       8
NORRLAND        8
Malmo           8
Linkoping       8
västerås        7
Norr            7
VÄSTERÅS        7
malmö           7
Name: count, dtype: int64

In [39]:
df_raw["kundtyp"].value_counts()

kundtyp
Privat       1535
Företag       783
privat         66
b2c            64
Konsument      60
PRIVAT         58
B2C            45
B2B            39
b2b            35
FÖRETAG        29
Firma          27
företag        26
Name: count, dtype: int64

In [40]:
df_raw["betalmetod"].value_counts()

betalmetod
Faktura           938
Kort              775
Swish             550
Invoice            51
FAKTURA            50
faktura            44
Kreditkort         36
KORT               33
SWISH              31
swish              31
kort               30
Mobilbetalning     29
Visa               28
Mastercard         25
Name: count, dtype: int64

In [ ]:
df_raw["leveransstatus"].value_counts()

leveransstatus
Levererad          2005
Under transport     152
Retur               132
Skickad              95
Mottagen             92
levererad            86
LEVERERAD            68
Returnerad           10
Återsänd              8
På väg                7
retur                 6
under transport       6
RETUR                 4
UNDER TRANSPORT       2
Name: count, dtype: int64

In [25]:
df_raw["orderrad_id"].isna().sum()


np.int64(0)

In [ ]:
df_raw.duplicated().sum()

np.int64(67)

In [27]:
df_raw["orderrad_id"].duplicated().sum()


np.int64(67)

In [31]:
df_raw["order_id"].duplicated().sum()

np.int64(1110)

In [29]:
dupes = df_raw[df_raw["orderrad_id"].duplicated(keep=False)].sort_values("orderrad_id")
dupes.head(10)


,order_id,orderrad_id,orderdatum,leveransdatum,produkt_sku,produktnamn,kategori,antal,pris_per_enhet,region,kundtyp,betalmetod,kund_id,leveransstatus,recension_text,recensionsdatum,betyg
613,ORD-2024-00070,ORD-2024-00070-1,2024-11-18,2024-11-20,SKU-KB002,Kompakt Tangentbord Mini,Tillbehör,1,599.00,Stockholm,Privat,Kort,KND-13904,Levererad,Riktigt nöjd! Använder den dagligen.,2024-11-23,4.0
107,ORD-2024-00070,ORD-2024-00070-1,2024-11-18,2024-11-20,SKU-KB002,Kompakt Tangentbord Mini,Tillbehör,1,599.00,Stockholm,Privat,Kort,KND-13904,Levererad,Riktigt nöjd! Använder den dagligen.,2024-11-23,4.0
181,ORD-2024-00110,ORD-2024-00110-1,2024-07-11,2024-07-14,SKU-MS001,Trådlös Mus X1,Tillbehör,2,399 kr,Gbg,privat,Kort,KND-13544,Levererad,"Leverans på två dagar, imponerad!",2024-07-21,4.0
1403,ORD-2024-00110,ORD-2024-00110-1,2024-07-11,2024-07-14,SKU-MS001,Trådlös Mus X1,Tillbehör,2,399 kr,Gbg,privat,Kort,KND-13544,Levererad,"Leverans på två dagar, imponerad!",2024-07-21,4.0
2003,ORD-2024-00111,ORD-2024-00111-1,2024-06-21,2024-06-26,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,NORRLAND,Företag,Faktura,KND-39076,Levererad,Leveransskada - kartongen var helt demolerad.,2024-07-06,2.0
182,ORD-2024-00111,ORD-2024-00111-1,2024-06-21,2024-06-26,SKU-MN002,"Bildskärm 24"" FHD",Bildskärmar,1,2499.00,NORRLAND,Företag,Faktura,KND-39076,Levererad,Leveransskada - kartongen var helt demolerad.,2024-07-06,2.0
224,ORD-2024-00138,ORD-2024-00138-2,2024-09-17,2024-09-21,SKU-HS002,Headset Budget,Ljud,2,499.00,Linköping,b2b,Kort,KND-41713,Levererad,Fungerar inte som det ska. Måste returnera.,2024-09-29,1.0
1682,ORD-2024-00138,ORD-2024-00138-2,2024-09-17,2024-09-21,SKU-HS002,Headset Budget,Ljud,2,499.00,Linköping,b2b,Kort,KND-41713,Levererad,Fungerar inte som det ska. Måste returnera.,2024-09-29,1.0
276,ORD-2024-00168,ORD-2024-00168-2,2024-07-31,2024-08-02,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,Göteborg,privat,Faktura,KND-30388,Levererad,Prisvärt och bra kvalitet. Kommer köpa igen.,2024-08-14,5.0
383,ORD-2024-00168,ORD-2024-00168-2,2024-07-31,2024-08-02,SKU-MS002,Ergonomisk Mus Pro,Tillbehör,1,699.00,Göteborg,privat,Faktura,KND-30388,Levererad,Prisvärt och bra kvalitet. Kommer köpa igen.,2024-08-14,5.0


In [30]:
(df_raw[["order_id", "orderrad_id"]]
 .duplicated()
 .sum())


np.int64(67)

In [32]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2767 entries, 0 to 2766
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         2767 non-null   str    
 1   orderrad_id      2767 non-null   str    
 2   orderdatum       2767 non-null   str    
 3   leveransdatum    2767 non-null   str    
 4   produkt_sku      2767 non-null   str    
 5   produktnamn      2767 non-null   str    
 6   kategori         2767 non-null   str    
 7   antal            2767 non-null   str    
 8   pris_per_enhet   2767 non-null   str    
 9   region           2612 non-null   str    
 10  kundtyp          2767 non-null   str    
 11  betalmetod       2651 non-null   str    
 12  kund_id          2767 non-null   str    
 13  leveransstatus   2673 non-null   str    
 14  recension_text   1355 non-null   str    
 15  recensionsdatum  1355 non-null   str    
 16  betyg            1355 non-null   float64
dtypes: float64(1), str(16)
me

Frågeställningar : Se vilka produkter som säljs mest samt vilka kategorier. Se vilken kategori som genererar mest netto. 

# EDA

1. Datasetets grain är på orderrad-nivå, men det finns dubletter som behöver hanteras.

2. Orderdatum är en sträng som behöver göras om till datetime, samma gäller för leveransdatum. 

3. Antal ska ändras om till en int. 

4. Pris_per_enhet ska göras om till float, samt standardisera värdet i pris_per_enhet.

5. Region beöver standardiseras och nullvärden hanteras. 

6. Kundtyp behöver standardiseras till rätt namn. 

7. betalmetod behöver standardiseras till rätt namn, samt hantera nullvärden.

8. Leveransstatus behöver standardiseras till rätt namn, samt hantera nullvärden.

9. Behöver göra om recensionsdatum till datetime. 

In [ ]:
#1. fixa dubletter i orderrader
def clean_orderrader(df):
    df_clean = df.copy()
    # Ta bort dubbletter baserat på 'orderrad_id'
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    after = len(df_clean)

    print(f"Tagit bort dubletter: {before - after}")

    return df_clean



Tagit bort dubletter: 67


In [96]:
#2. fixa orderdatum
def clean_orderdatum(df):
    df_clean = df.copy()

    # 1) Normalisera text (svenska månadsnamn -> engelska) + ta bort kommatecken
    month_map = {
        "januari": "january",
        "februari": "february",
        "mars": "march",
        "april": "april",
        "maj": "may",
        "juni": "june",
        "juli": "july",
        "augusti": "august",
        "september": "september",
        "oktober": "october",
        "november": "november",
        "december": "december",
    }

    s = df_clean["orderdatum"].astype("string").str.strip().str.lower()
    s = s.str.replace(",", "", regex=False)

    # Byt ut svenska månadsnamn oavsett var de ligger i strängen
    for sv, en in month_map.items():
        s = s.str.replace(rf"\b{sv}\b", en, regex=True)

    # 2) Parse till datetime
    df_clean["orderdatum"] = pd.to_datetime(
        s,
        errors="coerce",
        dayfirst=True,
        format="mixed",
    )

    invalid_dates = df_clean["orderdatum"].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'orderdatum', sätter dessa till NaT.")
    return df_clean

df_clean = clean_orderdatum(df)

print(df_clean["orderdatum"].isna().sum())
print(df_clean["orderdatum"].dtype)

0
datetime64[us]


Här gjorde jag om orderdatum till datetime och eftersom att det fanns olika format och även bokstäver i samt på både svenska och engelska så fick jag standardisera språket sedan lägga in regex för att den skulle kunna konvertera om vart än månaden var skriven i texten. 

In [95]:
#2.1 fixa leveransdatum
def clean_leveransdatum(df):
    df_clean = df.copy()

    # 1) Normalisera text (svenska månadsnamn -> engelska) + ta bort kommatecken
    month_map = {
        "januari": "january",
        "februari": "february",
        "mars": "march",
        "april": "april",
        "maj": "may",
        "juni": "june",
        "juli": "july",
        "augusti": "august",
        "september": "september",
        "oktober": "october",
        "november": "november",
        "december": "december",
    }

    s = df_clean["leveransdatum"].astype("string").str.strip().str.lower()
    s = s.str.replace(",", "", regex=False)

    # Byt ut svenska månadsnamn oavsett var de ligger i strängen
    for sv, en in month_map.items():
        s = s.str.replace(rf"\b{sv}\b", en, regex=True)

    # 2) Parse till datetime
    df_clean["leveransdatum"] = pd.to_datetime(
        s,
        errors="coerce",
        dayfirst=True,
        format="mixed",
    )

    invalid_dates = df_clean["leveransdatum"].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'leveransdatum', sätter dessa till NaT.")

    return df_clean

df_clean = clean_leveransdatum(df)

print(df_clean["leveransdatum"].isna().sum())
print(df_clean["leveransdatum"].dtype)



0
datetime64[us]


Här gjorde jag om orderdatum till datetime och eftersom att det fanns olika format och även bokstäver i samt på både svenska och engelska så fick jag standardisera språket sedan lägga in regex för att den skulle kunna konvertera om vart än månaden var skriven i texten. 

In [ ]:
#3.
def clean_antal(df):
    df_clean = df.copy()
    
    # Konvertera 'antal' till numeriskt format
    df_clean['antal'] = pd.to_numeric(
        df_clean['antal'], 
        errors='coerce')

    # Hantera negativa och nollvärden
    invalid_antal = (df_clean['antal'] <= 0).sum()
    if invalid_antal > 0:
        print(f"Hittade {invalid_antal} ogiltiga värden i 'antal' (negativa eller noll), sätter dessa till NaN.")
        
        df_clean.loc[df_clean['antal'] <= 0, 'antal'] = np.nan

    df_clean["antal"] = df_clean["antal"].astype("Int64")

    return df_clean



Int64


Här gjorde jag om antal från en sträng till int64.

In [90]:
#4.
def clean_pris_per_enhet(df):
    df_clean = df.copy()

    df_clean['pris_per_enhet'] = (
        df_clean['pris_per_enhet']
        .astype(str)
        .str.replace(",", ".", regex=False)              # hantera decimal-komma
        .str.replace(r"[^0-9.]", "", regex=True)
    )
    
    # Konvertera 'pris_per_enhet' till numeriskt format
    df_clean['pris_per_enhet'] = pd.to_numeric(
        df_clean['pris_per_enhet'],
        errors='coerce')

    # Hantera negativa och nollvärden
    invalid_pris = (df_clean['pris_per_enhet'] <= 0).sum()
    if invalid_pris > 0:
        print(f"Hittade {invalid_pris} ogiltiga värden i 'pris_per_enhet' (negativa eller noll), sätter dessa till NaN.")
        
        df_clean.loc[df_clean['pris_per_enhet'] <= 0, 'pris_per_enhet'] = np.nan

    return df_clean




I pris_per_enhet fanns en del fel så jag fick först standardisera hur jag ville att det skulle se ut. så jag valde att ta bort att bokstäver och ",". Sedan göra om strängen till float för att enklare kunna göra beräkningar senare. 

In [ ]:
#5.
def standardisera_region(df):
    df_clean = df.copy()
    df_clean["region_raw"] = df_clean["region"]

    def klassificera_region(val):
        if pd.isna(val):
            return "Okänd"

        v = str(val).lower().strip()

        if v in ["stockholm", "sthlm", "sthl", "sthml"]:
            return "Stockholm"

        if v in ["uppsala"]:
            return "Uppsala"

        if v in ["göteborg", "gothenburg", "gbg", "gbgb"]:
            return "Göteborg"

        if v in ["malmö", "malmo"]:
            return "Malmö"

        if v in ["norrland", "norr"]:
            return "Norrland"

        if v in ["örebro", "orebro"]:
            return "Örebro"

        if v in ["västerås", "vasteras"]:
            return "Västerås"

        if v in ["linköping", "linkoping"]:
            return "Linköping"

        return "Okänd"

    df_clean["region"] = df_clean["region"].apply(klassificera_region)
    return df_clean




region
Stockholm    1034
Göteborg      524
Malmö         279
Uppsala       238
Okänd         155
Norrland      149
Örebro        145
Linköping     144
Västerås       99
Name: count, dtype: int64


I region tabellen så hade vi mycket olika stavningar och förkortningar. Så här gjorde jag först om det till lower och tog bort whitspace och sedan kunde jag skriva "om detta finns i detta gör det till detta. om inte gör det till "okänd". Mest för att slippa hårkoda in alla typer av olika stora och små bokstäver som skrivs. 

In [ ]:
#6.
def standardisera_kundtyp(df):
    df_clean = df.copy()
    df_clean["kundtyp_raw"] = df_clean["kundtyp"]

    def klassificera_kundtyp(val):
        if pd.isna(val):
            return "Okänd"

        v = str(val).lower().strip()

        # Privat / B2C
        if any(x in v for x in ["privat", "b2c", "konsument"]):
            return "Privat"

        # Företag / B2B
        if any(x in v for x in ["företag", "firma", "b2b"]):
            return "Företag"

        return "Okänd"

    df_clean["kundtyp"] = df_clean["kundtyp"].apply(klassificera_kundtyp)
    return df_clean




kundtyp
Privat     1828
Företag     939
Name: count, dtype: int64


Här gjorde jag samma som i den tidigare funktionen. 

In [ ]:
#7. 
def standardisera_betalmetod(df):
    df_clean = df.copy()
    df_clean["betalmetod_raw"] = df_clean["betalmetod"]

    def klassificera_betalmetod(val):
        if pd.isna(val):
            return "Okänd"

        v = str(val).lower().strip()

        # Faktura
        if any(x in v for x in ["faktura", "invoice"]):
            return "Faktura"

        # Swish
        if "swish" in v:
            return "Swish"

        # Kortbetalning
        if any(x in v for x in ["kort", "visa", "mastercard", "kredit"]):
            return "Kort"

        # Mobilbetalning
        if "mobil" in v:
            return "Mobilbetalning"

        return "Okänd"

    df_clean["betalmetod"] = df_clean["betalmetod"].apply(klassificera_betalmetod)
    return df_clean





betalmetod
Faktura           1083
Kort               927
Swish              612
Okänd              116
Mobilbetalning      29
Name: count, dtype: int64


Jag gjorde samma som i de 2 tidigare funktionerna men i denna så såg jag även att mobiltelefon utger en väldigt liten del vilket gör att när jag gör beräkningar så kommer jag inte att ta med den för den kommer göra beräkningen dålig. 

In [94]:
#8.
def standardisera_leveransstatus(df):
    df_clean = df.copy()

    # Grundläggande textnormalisering
    status = (
        df_clean['leveransstatus']
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_clean['leveransstatus_std'] = np.select(
        [
            status.str.contains(r"levererad|mottagen"),
            status.str.contains(r"transport|skickad|på väg"),
            status.str.contains(r"retur|return|återsänd"),
        ],
        [
            "Levererad",
            "Under transport",
            "Retur",
        ],
        default="Okänd"
    )

    return df_clean

df = standardisera_leveransstatus(df)

df['leveransstatus_std'].value_counts()


leveransstatus_std
Levererad          2251
Under transport     262
Retur               160
Okänd                94
Name: count, dtype: int64

Denna funktionen så standardiserade jag leveransstatus. Även i detta fall valde jag att inte använda en hårdkodad mapping. Men tog hjälp av rexex att begränsa till 4 kategorier och klassificeringen gjordes vektoriserat med hjälp av np.

In [97]:
#9.
def clean_recensionsdatum(df):
    df_clean = df.copy()

    # 1) Normalisera text (svenska månadsnamn -> engelska) + ta bort kommatecken
    month_map = {
        "januari": "january",
        "februari": "february",
        "mars": "march",
        "april": "april",
        "maj": "may",
        "juni": "june",
        "juli": "july",
        "augusti": "august",
        "september": "september",
        "oktober": "october",
        "november": "november",
        "december": "december",
    }

    s = df_clean["recensionsdatum"].astype("string").str.strip().str.lower()
    s = s.str.replace(",", "", regex=False)

    # Byt ut svenska månadsnamn oavsett var de ligger i strängen
    for sv, en in month_map.items():
        s = s.str.replace(rf"\b{sv}\b", en, regex=True)

    # 2) Parse till datetime
    df_clean["recensionsdatum"] = pd.to_datetime(
        s,
        errors="coerce",
        dayfirst=True,
        format="mixed",
    )

    invalid_dates = df_clean["recensionsdatum"].isna().sum()
    if invalid_dates > 0:
        print(f"Hittade {invalid_dates} ogiltiga datum i 'recensionsdatum', sätter dessa till NaT.")
    return df_clean

df_clean = clean_recensionsdatum(df)

print(df_clean["recensionsdatum"].isna().sum())
print(df_clean["recensionsdatum"].dtype)

Hittade 1412 ogiltiga datum i 'recensionsdatum', sätter dessa till NaT.
1412
datetime64[us]


Även i recensionsdatum fanns samma fel i hur datumen var skrivna så använde mig av samma funktion men bytte ut kolumnnamnet. 1412 ogiltiga datum finns för att det var så många saknade rader i den kolumnen. 

# Skapa data-dictionary

In [102]:

# Dokumentation baserad på min EDA
COLUMN_DOCS = {
    "order_id": {
        "description": "Unikt ID för varje order.",
        "allowed_values_or_format": "Integer",
        "rules_and_notes": "Kan förekomma flera gånger eftersom datasetet är på orderrad-nivå."
    },
    "orderrad_id": {
        "description": "Unikt ID för varje orderrad.",
        "allowed_values_or_format": "Integer",
        "rules_and_notes": "Identifierar datasetets grain (orderrad-nivå). Dubletter identifierades och hanterades."
    },
    "orderdatum": {
        "description": "Datum då ordern skapades.",
        "allowed_values_or_format": "datetime",
        "rules_and_notes": "Konverterad från sträng till datetime. Blandade datumformat (svenska/engelska månadsnamn) standardiserades."
    },
    "leveransdatum": {
        "description": "Datum då ordern levererades.",
        "allowed_values_or_format": "datetime",
        "rules_and_notes": "Konverterad från sträng till datetime. Ogiltiga datum sattes till NaT."
    },
    "antal": {
        "description": "Antal produkter på orderraden.",
        "allowed_values_or_format": "Integer ≥ 1",
        "rules_and_notes": "Konverterad till int för korrekt beräkning av intäkter."
    },
    "pris_per_enhet": {
        "description": "Pris per enhet för produkten på orderraden.",
        "allowed_values_or_format": "Float, t.ex. 1200.00",
        "rules_and_notes": "Rensad från bokstäver, valutor och symboler samt standardiserad till numeriskt format."
    },
    "region": {
        "description": "Geografisk region kopplad till ordern.",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad till konsekventa regionnamn. Saknade värden hanterades."
    },
    "kundtyp": {
        "description": "Typ av kund (t.ex. privat eller företag).",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad till konsekventa benämningar."
    },
    "betalmetod": {
        "description": "Betalningsmetod som användes vid köpet.",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad till konsekventa namn. Nullvärden hanterades."
    },
    "leveransstatus": {
        "description": "Leveransstatus för ordern.",
        "allowed_values_or_format": "Kategorisk text",
        "rules_and_notes": "Standardiserad via regelbaserad textklassificering (t.ex. Levererad, Under transport, Retur)."
    },
    "recensionsdatum": {
        "description": "Datum då recensionen lämnades.",
        "allowed_values_or_format": "datetime",
        "rules_and_notes": "Konverterad från sträng till datetime. Ogiltiga värden sattes till NaT."
    }
}

def skapa_data_dictionary(df_clean: pd.DataFrame, docs: dict) -> pd.DataFrame:
    rows = []
    n = len(df_clean)

    for col in df_clean.columns:
        s = df_clean[col]
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "non_null_pct": round(s.notna().mean() * 100, 1),
            "n_unique": s.nunique(dropna=True),
            "example_value": s.dropna().iloc[0] if s.dropna().shape[0] > 0 else np.nan,
            "description": docs.get(col, {}).get("description", ""),
            "allowed_values_or_format": docs.get(col, {}).get("allowed_values_or_format", ""),
            "rules_and_notes": docs.get(col, {}).get("rules_and_notes", ""),
        })

    return pd.DataFrame(rows).sort_values("column").reset_index(drop=True)

# Skapa data dictionary
data_dict = skapa_data_dictionary(df_clean, COLUMN_DOCS)

# Spara som CSV
data_dict.to_csv("ehandel_data_dictionary.csv", index=False, encoding="utf-8")

print("✓ Sparade: ehandel_data_dictionary.csv")
print("\nData dictionary:")
data_dict



✓ Sparade: ehandel_data_dictionary.csv

Data dictionary:


,column,dtype,non_null_pct,n_unique,example_value,description,allowed_values_or_format,rules_and_notes
0,antal,Int64,94.6,5,1,Antal produkter på orderraden.,Integer ≥ 1,Konverterad till int för korrekt beräkning av ...
1,betalmetod,str,95.8,14,Kort,Betalningsmetod som användes vid köpet.,Kategorisk text,Standardiserad till konsekventa namn. Nullvärd...
2,betyg,float64,49.0,5,2.0,,,
3,kategori,str,100.0,5,Tillbehör,,,
4,kund_id,str,100.0,1644,KND-53648,,,
5,kundtyp,str,100.0,12,Privat,Typ av kund (t.ex. privat eller företag).,Kategorisk text,Standardiserad till konsekventa benämningar.
6,leveransdatum,datetime64[us],100.0,368,2024-05-22 00:00:00,Datum då ordern levererades.,datetime,Konverterad från sträng till datetime. Ogiltig...
7,leveransstatus,str,96.6,14,Levererad,Leveransstatus för ordern.,Kategorisk text,Standardiserad via regelbaserad textklassifice...
8,leveransstatus_std,str,100.0,4,Levererad,,,
9,order_id,str,100.0,1657,ORD-2024-00001,Unikt ID för varje order.,Integer,Kan förekomma flera gånger eftersom datasetet ...
